In [ ]:
import random

# Shuffling image data before splitting it

random.seed(49328042)
images_shuffled = os.listdir("./datasets/faces/cropped")
random.shuffle(images_shuffled)

age_labels = []

for file in images_shuffled:
    age_label = int(file[0:file.index('_')])
    age_labels.append(age_label)


In [ ]:
# Split the data into training, validation, and testing sets

train_data = images_shuffled[:45000]
train_labels = age_labels[:45000]

validation_data = images_shuffled[45001:49000]
validation_labels = age_labels[45001:49000]

testing_data = images_shuffled[49001:]
testing_labels = age_labels[49001:]

In [ ]:
# Create image set directories based on labels

import os, shutil, pathlib

original_dir = "./" + str(pathlib.Path("datasets/faces/cropped"))
base_training_dir = "./" + str(pathlib.Path("datasets/faces/training"))

def make_data_sets(subdir, starting_index, ending_index):
    for img_index in range(starting_index, ending_index):
        dir_path = f"{base_training_dir}/{subdir}/{str(age_labels[img_index])}"
        os.makedirs(dir_path, exist_ok=True)

        image_path = os.path.join(original_dir, images_shuffled[img_index])
        if (image_path not in os.listdir(dir_path)):
            shutil.move(image_path, dir_path)

if (len(os.listdir(original_dir)) > 0):
    make_data_sets("train", 0, 45000)
    make_data_sets("validation", 45001, 49000)
    make_data_sets("test", 49001, 57000)



In [ ]:
# Build tf.data pipelines with labels parsed directly from file paths.
# Important: shuffle file paths (cheap), not decoded images (expensive).

import tensorflow as tf
import os
from pathlib import Path

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def decode_image_and_label(path, training=False):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, IMG_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32)

    # Parent folder name is the age label (e.g. .../train/34/image.jpg -> 34.0).
    age_str = tf.strings.split(path, os.sep)[-2]
    label = tf.strings.to_number(age_str, out_type=tf.float32)

    if training:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=0.08)
        image = tf.image.random_contrast(image, lower=0.9, upper=1.1)

    return image, label

def make_dataset(split_name, training=False):
    split_dir = Path(base_training_dir) / split_name
    pattern = str(split_dir / "*" / "*.jpg")

    # Create file-path dataset first, then shuffle paths (strings), not image tensors.
    file_ds = tf.data.Dataset.list_files(pattern, shuffle=False)
    if training:
        file_ds = file_ds.shuffle(50000, seed=42, reshuffle_each_iteration=True)

    ds = file_ds.map(
        lambda p: decode_image_and_label(p, training=training),
        num_parallel_calls=AUTOTUNE,
    )

    # Skip occasional corrupt files instead of crashing the input pipeline.
    ds = ds.apply(tf.data.experimental.ignore_errors())
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_dataset = make_dataset("train", training=True)
validation_dataset = make_dataset("validation", training=False)
test_dataset = make_dataset("test", training=False)

print("Datasets loaded with deterministic path-based labels.")
for imgs, labels in train_dataset.take(1):
    print(f"Batch images: {imgs.shape}, labels sample: {labels[:8].numpy()}")

In [ ]:
for data_batch, labels_batch in train_dataset:
    print(f"Shape of data set: {data_batch.shape}")
    print(f"Shape of label set: {labels_batch.shape}")
    break

### Building the CNN

In [ ]:
%pip install tensorflow
%pip install numpy

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Strong baseline for age regression: pretrained EfficientNet backbone + regression head.
inputs = keras.Input(shape=(224, 224, 3), name="image")

backbone = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_tensor=inputs,
    pooling=None,
 )
backbone.trainable = False  # Stage 1: train head first

x = keras.applications.efficientnet.preprocess_input(inputs)
x = backbone(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.20)(x)
outputs = layers.Dense(1, name="predicted_age")(x)

cnn_model = keras.Model(inputs, outputs, name="age_effnetb0_regressor")
cnn_model.summary()

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")

# Prevent TF from grabbing all GPU memory up front.
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Enabled memory growth on: {gpu}")
    except Exception as err:
        print(f"Could not set memory growth on {gpu}: {err}")

In [ ]:
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import register_keras_serializable

# Custom metric to track percent of age predictions correct within N-year tolerance.
@register_keras_serializable(package="Custom")
class WithinNYears(tf.keras.metrics.Metric):
    def __init__(self, tolerance=None, dtype=None, name=None, **kwargs):
        # Backward-compatible load path: if older saved configs omit tolerance
        if tolerance is None and isinstance(name, str):
            match = re.match(r"within_(\d+)_years", name)
            if match:
                tolerance = int(match.group(1))

        if tolerance is None:
            tolerance = 2

        super().__init__(name=name or f"within_{tolerance}_years", dtype=dtype, **kwargs)
        self.tolerance = int(tolerance)
        self.correct = self.add_weight(name="correct", initializer="zeros")
        self.total = self.add_weight(name="total", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.squeeze(y_pred, axis=-1) # get last tensor (output) and transform (batch_size,1) to (batch_size,)
        y_true = tf.cast(y_true, tf.float32) 
        within = tf.abs(y_true - y_pred) <= self.tolerance
        self.correct.assign_add(tf.reduce_sum(tf.cast(within, tf.float32))) # convert booleans to floats and get total sum and add to correct
        self.total.assign_add(tf.cast(tf.size(y_true), tf.float32)) # total is just size of y_true

    def result(self):
        return self.correct / self.total
    
    def reset(self): # runs once per epoch
        self.correct.assign(0)
        self.total.assign(0)

    def get_config(self):
        config = super().get_config()
        config.update({"tolerance": self.tolerance})
        return config

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Stage 1 optimizer/schedule (head training while backbone frozen).
lr_schedule = keras.callbacks.ReduceLROnPlateau(
    monitor="val_mae", factor=0.5, patience=2, min_lr=1e-6, verbose=1
)

cnn_model.compile(
    loss=keras.losses.Huber(delta=4.0),
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["mae", WithinNYears(2), WithinNYears(5), WithinNYears(10)]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="cnn_model_best.keras",
        save_best_only=True,
        monitor="val_mae",
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    lr_schedule,
]

print("Stage 1 model compiled.")

In [ ]:
list = [1,2,3,4,5,6]
print(list[:-3])

In [ ]:
# Stage 1: train regression head
history_stage1 = cnn_model.fit(
    train_dataset,
    epochs=12,
    validation_data=validation_dataset,
    callbacks=callbacks,
 )

# Stage 2: unfreeze top backbone layers and fine-tune at lower LR
backbone = cnn_model.get_layer("efficientnetb0")
backbone.trainable = True

# Keep early layers frozen, fine-tune only deeper layers
for layer in backbone.layers[:-40]:
    layer.trainable = False

cnn_model.compile(
    loss=keras.losses.Huber(delta=4.0),
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    metrics=["mae", WithinNYears(2), WithinNYears(5), WithinNYears(10)]
)

history_stage2 = cnn_model.fit(
    train_dataset,
    epochs=25,
    validation_data=validation_dataset,
    callbacks=callbacks,
 )

training_history = history_stage2

### Tuning Model

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    mae = history.history["mae"]
    val_mae = history.history["val_mae"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs = range(1, len(mae) + 1)

    plt.plot(epochs, mae, "bo", label="Training Mae")
    plt.plot(epochs, val_mae, "b", label="Validation Mae")
    plt.title("Training and Validation MAE")
    plt.xlabel("Epochs")
    plt.ylabel("MAE")
    plt.legend()
    plt.figure()

    plt.plot(epochs[1:], loss[1:], "bo", label="Training Loss")
    plt.plot(epochs[1:], val_loss[1:], "b", label="Validation Loss")
    plt.title("Training and Validation Loss")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


In [ ]:
plot_training_history(training_history)

### Evaluating the Model

In [ ]:
custom_objects = {"WithinNYears": WithinNYears}
test_model = keras.models.load_model("cnn_model_best.keras", custom_objects=custom_objects)
test_loss = test_model.evaluate(test_dataset)
print(f"Test Loss: {test_loss[0]:.3f}\nTest MAE: {test_loss[1]:.3f}\nTest within 10y: {test_loss[4]:.3f}\nTest within 5y: {test_loss[3]:.3f}\nTest within 2y: {test_loss[2]:.3f}")